<a href="https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srvnkumr27/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [51]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Retrieve Hugging Face token safely from environment or Colab secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.getenv('HF_TOKEN')

# 2. Connect DuckDB and authenticate with Hugging Face
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 3. Base path for mid-panel month (March 2026)
DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"
MID_PANEL_PATH = f"{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet"

print("DuckDB initialized with Hugging Face token successfully.")

DuckDB initialized with Hugging Face token successfully.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of Analysis + Time Window

* **Unit of Analysis (Grain):** One row represents daily aggregate search performance metrics for a single content item (`content_id`) belonging to a specific client (`client_id`) on a specific report date (`report_date`).
* **Time Window:** Mid-panel month `month=2026-03` (March 1, 2026 – March 31, 2026). The final month (`month=2026-06`) is intentionally excluded and reserved strictly as a sealed test month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Field Classification

| Field Name | Bucket | Classification & "Available When?" Justification |
| :--- | :--- | :--- |
| `clicks_7d_avg` | **Feature** | Knowable at decision moment $t$ because it uses strictly historical logs up to date $t$. |
| `impressions_7d_avg` | **Feature** | Knowable at decision moment $t$ because past impression traffic is recorded prior to cutoff. |
| `position_7d_avg` | **Feature** | Knowable at decision moment $t$ because historical SERP positions are logged prior to date $t$. |
| `ctr_7d` | **Feature** | Knowable at decision moment $t$ as a ratio derived from historic telemetry ($\frac{\text{clicks}_{7d}}{\text{impressions}_{7d}}$). |
| `is_weekend` | **Feature** | Knowable at decision moment $t$ because calendar day-of-week properties are known in advance. |
| `target_next_day_clicks` | **Label** | Target metric to predict (`LEAD(clicks)` on day $t+1$). |
| `client_id`, `content_id`, `report_date` | **Context** | Administrative primary keys defining the entity grain and time anchor. |
| `is_available` | **Excluded** | Used exclusively as a SQL filter (`is_available IS TRUE`). Non-available rows represent unverified logs. |
| `trap_next_day_impressions` | **Excluded (Trap)** | Excluded in honest models. Future impressions on day $t+1$ are unavailable at time $t$ and introduce data leakage. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [53]:
# --- Claim 1 Query: Grain Uniqueness ---
query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS unique_grain_keys,
    COUNT(*) - COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS duplicate_rows
FROM read_parquet('{MID_PANEL_PATH}');
"""
print("--- Grain Verification (Must show 0 duplicate rows) ---")
display(con.sql(query_grain).df())

--- Grain Verification (Must show 0 duplicate rows) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_keys,duplicate_rows
0,9841378,9841378,0


In [54]:
# --- Claim 2 Query: Row Count & Date Span ---
query_span = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT report_date) AS distinct_days_in_month
FROM read_parquet('{MID_PANEL_PATH}');
"""
print("--- Time Window Span Verification ---")
display(con.sql(query_span).df())

--- Time Window Span Verification ---


,total_rows,min_report_date,max_report_date,distinct_days_in_month
0,9841378,2026-03-01,2026-03-31,31


In [57]:
# --- Claim 3 Query: Availability and Missing Value Check ---
query_avail = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS available_rows,
    COUNT(CASE WHEN gsc_data_available IS NOT TRUE THEN 1 END) AS excluded_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_survived,
    COUNT(CASE WHEN gsc_clicks IS NULL THEN 1 END) AS null_clicks,
    COUNT(CASE WHEN gsc_impressions IS NULL THEN 1 END) AS null_impressions
FROM read_parquet('{MID_PANEL_PATH}');
"""
print("--- Availability Check (IS TRUE) & Null Counts ---")
display(con.sql(query_avail).df())

--- Availability Check (IS TRUE) & Null Counts ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,excluded_rows,pct_survived,null_clicks,null_impressions
0,9841378,3611061,6230317,36.69,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What This Data Can Never Tell You
While this dataset provides a structured daily view of search telemetry, structural boundaries in the warehouse mean it cannot answer certain critical questions:

Unbalanced History Depth Across Clients (Cold-Start Truncation)

The Limit: Clients onboarded mid-panel or with staggered historical sync dates (gsc_data_start, ga4_data_start) lack complete rolling historical windows.

Why it matters: For newer content or freshly integrated clients, a 7-day rolling average (ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) evaluates over incomplete windows (e.g., only 2–3 days of history), artificially deflating baseline features to near zero. The data cannot tell you whether a low 7-day average reflects true poor performance or simply missing historical logs prior to onboarding.

GSC-Only Early Telemetry (The Behavioral Blindspot)

The Limit: Historic reporting slices prior to full GA4 account linking capture top-of-funnel Search Console metrics (clicks, impressions, position), but contain zero user engagement data.

Why it matters: The model cannot measure post-click user intent or content quality (such as bounce rates, time-on-page, scroll depth, or downstream conversion events) for these early rows. It only sees search engine exposure, not whether the traffic yielded meaningful engagement.

Rolling Window Overlaps (Shared Variance & Autocorrelation)

The Limit: Consecutive daily rows for the same content item share 6 overlapping days of historical data within their rolling 7-day aggregations.

Why it matters: The dataset cannot be treated as independent, identically distributed (i.i.d.) observations. Standard random cross-validation will cause severe target leakage across adjacent train/test splits due to auto-correlated features. Evaluating performance requires time-grouped or block-wise validation strategies rather than standard random sampling.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.